In [ ]:
# Imports necessaires pour Kaggle
import pandas as pd
import numpy as np
import os
import zipfile
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import explode, col
from IPython.display import FileLink, display

print("Tous les imports sont charges")

In [ ]:
# Lecture du dataset depuis Kaggle
print("Lecture du dataset depuis Kaggle...")

csv_path = '/kaggle/input/datasets/oumniaelrhadrouf/dataser-als/dataset-ALS.csv'
df = pd.read_csv(csv_path)

print(f"Fichier charge depuis: {csv_path}")
print(f"\nApercu:")
print(f"   - Lignes: {df.shape[0]:,}")
print(f"   - Colonnes: {df.shape[1]}")
print(f"   - Noms des colonnes: {df.columns.tolist()}")

In [ ]:
# Apercu des donnees
print("=== Apercu des donnees ===")
print(df.head())

# Statistiques
print(f"\nStatistiques du dataset:")
print(f"   - Nombre de lignes : {df.shape[0]:,}")
print(f"   - Nombre de colonnes : {df.shape[1]}")
print(f"   - Dimensions : {df.shape}")

# Noms des colonnes
print(f"\nNoms des colonnes :")
print(f"   {df.columns.tolist()}")

# Types de donnees
print(f"\nTypes des colonnes :")
print(df.dtypes)

In [ ]:
# Renommer les colonnes
df_als = df.rename(columns={
    'reviewerID': 'user_id',
    'asin': 'product_id',
    'overall': 'rating'
})

# Convertir les types
df_als['user_id'] = df_als['user_id'].astype(str)
df_als['product_id'] = df_als['product_id'].astype(str)
df_als['rating'] = df_als['rating'].astype(float)

print("Pretraitement termine")
print(f"   - user_id type: {df_als['user_id'].dtype}")
print(f"   - product_id type: {df_als['product_id'].dtype}")
print(f"   - rating type: {df_als['rating'].dtype}")

In [ ]:
# Sauvegarder en Parquet dans /kaggle/working/
output_path = '/kaggle/working/dataset-als.parquet'
df_als.to_parquet(output_path, index=False)
print(f"Fichier Parquet cree: {output_path}")

In [ ]:
# Demarrer Spark
print("\nDemarrage de Spark...")

spark = SparkSession.builder \
    .appName("ALS_Training_Kaggle") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.network.timeout", "800s") \
    .getOrCreate()

print(f"Spark demarre - Version: {spark.version}")

In [ ]:
# Lire les donnees avec Spark
df_spark = spark.read.parquet(output_path)

print("\n=== Apercu des donnees Spark ===")
df_spark.show(5)

print("\nSchema des donnees:")
df_spark.printSchema()

In [ ]:
# Encodage des IDs
print("\nEncodage des IDs...")

# Supprimer les colonnes existantes si necessaire
for col_name in ['user_id_int', 'product_id_int']:
    if col_name in df_spark.columns:
        df_spark = df_spark.drop(col_name)
        print(f"   Colonne {col_name} supprimee")

# Encoder les user_id
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_id_int", handleInvalid="keep")
df_spark = user_indexer.fit(df_spark).transform(df_spark)

# Encoder les product_id
product_indexer = StringIndexer(inputCol="product_id", outputCol="product_id_int", handleInvalid="keep")
df_spark = product_indexer.fit(df_spark).transform(df_spark)

# Convertir en entier
df_spark = df_spark.withColumn("user_id_int", df_spark["user_id_int"].cast(IntegerType()))
df_spark = df_spark.withColumn("product_id_int", df_spark["product_id_int"].cast(IntegerType()))

print("Encodage termine")

print("\nApres encodage:")
df_spark.select("user_id", "user_id_int", "product_id", "product_id_int", "rating").show(5)

In [ ]:
# Split train/test
train_df, test_df = df_spark.randomSplit([0.8, 0.2], seed=42)

print(f"\nSplit des donnees:")
print(f"   - Train count : {train_df.count():,}")
print(f"   - Test count  : {test_df.count():,}")
print(f"   - Total       : {train_df.count() + test_df.count():,}")

In [ ]:
# Definir l'evaluateur
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# Configuration ALS
als = ALS(
    userCol="user_id_int",
    itemCol="product_id_int",
    ratingCol="rating",
    rank=10,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop"
)

print("\nConfiguration ALS:")
print(f"   - userCol    : {als.getUserCol()}")
print(f"   - itemCol    : {als.getItemCol()}")
print(f"   - ratingCol  : {als.getRatingCol()}")
print(f"   - rank       : {als.getRank()}")
print(f"   - maxIter    : {als.getMaxIter()}")
print(f"   - regParam   : {als.getRegParam()}")

In [ ]:
# Entrainement
print("\nEntrainement du modele ALS...")
print("Cela peut prendre 20-30 minutes...")

model = als.fit(train_df)

print("Modele entraine avec succes!")

In [ ]:
# Predictions sur test
print("\nEvaluation du modele...")
predictions = model.transform(test_df)

# Calcul du RMSE
rmse = evaluator.evaluate(predictions)

print(f"\n{'='*50}")
print(f"RESULTAT FINAL")
print(f"{'='*50}")
print(f"RMSE sur test : {rmse:.4f}")
print(f"{'='*50}")

In [ ]:
# Analyse des predictions
print("\nAnalyse des predictions:")
predictions.select("rating", "prediction").describe().show()

# Compter les predictions valides
valid_predictions = predictions.filter(predictions.prediction.isNotNull()).count()
total_predictions = predictions.count()

print(f"\nPredictions valides : {valid_predictions:,} / {total_predictions:,} ({valid_predictions/total_predictions*100:.1f}%)")

In [ ]:
# Generation des recommandations pour TOUS les utilisateurs
print("\nGeneration des recommandations pour tous les utilisateurs...")
print("ATTENTION: Cette operation peut prendre 2 a 4 heures pour 6.7M lignes")

# Utiliser recommendForAllUsers pour tous les utilisateurs
user_recommendations = model.recommendForAllUsers(10)

print("Generation terminee")

print("\nTop 5 recommandations (IDs encodes):")
user_recommendations.show(5, truncate=False)

# Sauvegarder les recommandations en parquet
output_parquet = '/kaggle/working/recommendations_all_users.parquet'
user_recommendations.write.mode("overwrite").parquet(output_parquet)
print(f"Recommandations sauvegardees dans {output_parquet}")

# Compter le nombre total de recommandations
total_users = user_recommendations.count()
print(f"Nombre total d'utilisateurs avec recommandations: {total_users:,}")

In [17]:
# CONVERSION PARQUET → CSV 
import os
import pandas as pd
from IPython.display import FileLink, display

print("="*60)
print("CONVERSION DES RECOMMANDATIONS EN CSV")
print("="*60)

# 1. Lire le dossier Parquet avec Spark
parquet_dir = '/kaggle/working/recommendations_all_users.parquet'
print(f"\n Lecture du dossier: {parquet_dir}")

df_spark = spark.read.parquet(parquet_dir)

# 2. Compter le nombre d'utilisateurs
total_users = df_spark.count()
print(f" Nombre total d'utilisateurs: {total_users:,}")

# 3. Convertir en Pandas (TOUTES les données)
print("\n Conversion en Pandas (cela peut prendre du temps)...")
df_pandas = df_spark.toPandas()

print(f" Conversion terminée!")
print(f" Lignes: {len(df_pandas):,}")
print(f" Colonnes: {len(df_pandas.columns)}")

# 4. Sauvegarder en CSV
csv_path = '/kaggle/working/recommendations_complete.csv'
df_pandas.to_csv(csv_path, index=False)

# 5. Vérifier la taille du fichier
size_mb = os.path.getsize(csv_path) / (1024*1024)
print(f"\n Fichier CSV créé: recommendations_complete.csv")
print(f" Taille: {size_mb:.2f} MB")

# 6. Afficher un aperçu
print(f"\n Aperçu des 5 premières lignes:")
print(df_pandas.head())



CONVERSION DES RECOMMANDATIONS EN CSV

📂 Lecture du dossier: /kaggle/working/recommendations_all_users.parquet


📊 Nombre total d'utilisateurs: 728,634

🔄 Conversion en Pandas (cela peut prendre du temps)...


✅ Conversion terminée!
📊 Lignes: 728,634
📊 Colonnes: 2

📁 Fichier CSV créé: recommendations_complete.csv
📦 Taille: 380.27 MB

📋 Aperçu des 5 premières lignes:
   user_id_int                                    recommendations
0            1  [(149013, 6.999699115753174), (136431, 6.52173...
1            3  [(149013, 7.334928512573242), (136431, 6.57221...
2            7  [(149013, 6.87989616394043), (118640, 5.999993...
3           11  [(112600, 6.328299045562744), (149013, 6.23221...
4           26  [(149013, 7.469087600708008), (121118, 6.72107...

📥 TÉLÉCHARGEMENT:
Cliquez sur le lien ci-dessous pour télécharger le fichier CSV sur votre PC:


/kaggle/working/recommendations_complete.csv


✅ Fait! Le fichier va se télécharger sur votre ordinateur


In [24]:
# REFORMATAGE  (API)
print("Reformatage des recommandations pour l'API...")

import pandas as pd
import json

# Lire votre CSV
df = pd.read_csv('/kaggle/working/recommendations_complete.csv')

# Fonction pour extraire les product_id et ratings
def extract_recommendations(row_data):
    """Convertit les Row() en format JSON lisible"""
    # Nettoyer la chaîne
    data_str = str(row_data)
    
    # Extraire les paires (product_id, rating)
    import re
    pattern = r'product_id_int=(\d+), rating=([\d.]+)'
    matches = re.findall(pattern, data_str)
    
    recommendations = []
    for product_id, rating in matches[:10]:  # Top 10
        recommendations.append({
            "product_id": int(product_id),
            "predicted_rating": float(rating)
        })
    return recommendations

# Appliquer la transformation
df['recommendations_clean'] = df['recommendations'].apply(extract_recommendations)

# Garder seulement les colonnes utiles
df_final = df[['user_id_int', 'recommendations_clean']]

# Sauvegarder en format JSON (plus facile pour l'API)
df_final.to_json('/kaggle/working/recommendations_api.json', orient='records')

# Aussi en CSV lisible
df_final['recommendations_str'] = df_final['recommendations_clean'].apply(json.dumps)
df_final[['user_id_int', 'recommendations_str']].to_csv('/kaggle/working/recommendations_api.csv', index=False)

print(" Fichiers reformatés:")
print("   - recommendations_api.json (pour l'API)")
print("   - recommendations_api.csv (pour consultation)")

Reformatage des recommandations pour l'API...
✅ Fichiers reformatés:
   - recommendations_api.json (pour l'API)
   - recommendations_api.csv (pour consultation)


In [28]:
# CREATION DES MAPPINGS IDs
print("\nCreation des mappings IDs...")

# Mapping utilisateurs
user_mapping = train_df.select("user_id", "user_id_int").distinct().toPandas()
user_mapping.to_csv('/kaggle/working/user_mapping.csv', index=False)
print("user_mapping.csv (" + str(len(user_mapping)) + " utilisateurs)")

# Mapping produits
product_mapping = train_df.select("product_id", "product_id_int").distinct().toPandas()
product_mapping.to_csv('/kaggle/working/product_mapping.csv', index=False)
print("product_mapping.csv (" + str(len(product_mapping)) + " produits)")


Creation des mappings IDs...


26/05/26 02:21:46 WARN DAGScheduler: Broadcasting large task binary with size 32.7 MiB
26/05/26 02:22:02 WARN DAGScheduler: Broadcasting large task binary with size 32.7 MiB


user_mapping.csv (728634 utilisateurs)


26/05/26 02:22:12 WARN DAGScheduler: Broadcasting large task binary with size 32.7 MiB
26/05/26 02:22:27 WARN DAGScheduler: Broadcasting large task binary with size 32.7 MiB


product_mapping.csv (160042 produits)


In [29]:
# MAPPINGS IDs EN FORMAT JSON
print("\nCreation des mappings IDs en JSON...")

# Mapping utilisateurs
user_mapping = train_df.select("user_id", "user_id_int").distinct().toPandas()
user_mapping_dict = dict(zip(user_mapping['user_id'], user_mapping['user_id_int']))

import json
with open('/kaggle/working/user_mapping.json', 'w') as f:
    json.dump(user_mapping_dict, f)
print("user_mapping.json (" + str(len(user_mapping_dict)) + " utilisateurs)")

# Mapping produits
product_mapping = train_df.select("product_id", "product_id_int").distinct().toPandas()
product_mapping_dict = dict(zip(product_mapping['product_id_int'], product_mapping['product_id']))

with open('/kaggle/working/product_mapping.json', 'w') as f:
    json.dump(product_mapping_dict, f)
print("product_mapping.json (" + str(len(product_mapping_dict)) + " produits)")


Creation des mappings IDs en JSON...


26/05/26 02:27:40 WARN DAGScheduler: Broadcasting large task binary with size 32.7 MiB
26/05/26 02:27:53 WARN DAGScheduler: Broadcasting large task binary with size 32.7 MiB


user_mapping.json (728634 utilisateurs)


26/05/26 02:28:00 WARN DAGScheduler: Broadcasting large task binary with size 32.7 MiB
26/05/26 02:28:13 WARN DAGScheduler: Broadcasting large task binary with size 32.7 MiB


product_mapping.json (160042 produits)


In [ ]:
# Sauvegarder le modèle ALS 
print("Sauvegarde du modèle ALS...")

# Le modèle vient de votre entraînement
# model = als.fit(train_df)  

# Sauvegarder
model.write().overwrite().save('/kaggle/working/als_model')

print(" Modèle sauvegardé dans /kaggle/working/als_model")

In [ ]:
# Compresser le dossier als_model en ZIP
import zipfile
import os

print("Compression du modèle...")

model_path = '/kaggle/working/als_model'
zip_path = '/kaggle/working/als_model.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(model_path):
        for file in files:
            file_path = os.path.join(root, file)
            # Garder la structure du dossier
            arcname = os.path.relpath(file_path, '/kaggle/working/')
            zipf.write(file_path, arcname)
            print(f"   Ajouté: {arcname}")

# Vérifier la taille
size_mb = os.path.getsize(zip_path) / (1024*1024)
print(f"\n ZIP créé: als_model.zip")
print(f" Taille: {size_mb:.2f} MB")